# Question 9 - 55. Jump Game

You are given an integer array `nums`. You are initially positioned at the array's **first index**, and each element in the array represents your **maximum** jump length at that position.

Return `true` if you can reach the last index, or `false` otherwise.

**Example 1:**

    Input: nums = [2,3,1,1,4]
    Output: true
    Explanation: Jump 1 step from index 0 to 1, then 3 steps to the last index.

**Example 2:**

    Input: nums = [3,2,1,0,4]
    Output: false
    Explanation: You will always arrive at index 3 no matter what. Its maximum jump length is 0, which makes it impossible to reach the last index.

**Constraints:**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 10^5`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy: track the farthest index you can reach

📘 **Revise first:** [Pattern C · running best-so-far](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (try every jump) | O(2ⁿ) | O(n) |
| 1½ DP (is each index 'good'?) | O(n²) | O(n) |
| 2️⃣ Optimized (farthest reach) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

You're standing on the first square of a row of stepping stones. Each stone has a number: **the most squares you may jump forward from it**. You can always jump *fewer*. Can you reach the **last** stone?

```
nums = [2, 3, 1, 1, 4]
        ↑ from here you may jump 1 or 2 squares
```
From index 0, jump 1 to index 1. From there you may jump up to 3, which lands exactly on the last index → **true**.

```
nums = [3, 2, 1, 0, 4]
```
Every route ends up on index 3, which says `0`. You're stuck there and can never reach index 4 → **false**.

**Everyday picture: a growing "reachable zone".** Imagine painting every stone you can reach. Start with stone 0 painted. Walk left to right over the painted stones; from each one, paint everything it can reach. Because you can always jump *shorter*, the painted area is always **one solid stretch from the start**, with no gaps. So you only need to remember **how far the paint goes**. If you ever arrive at a stone that isn't painted, you're stuck.

### Step 0 · Clarify before coding

🗣️ *"Each value is the maximum jump length, so from index i I can land anywhere from i+1 up to i+nums[i]. I just need yes or no: can I reach the last index? I don't need the number of jumps."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Is the value a **maximum** (shorter jumps allowed)? | **Yes.** | That's why the reachable area has no gaps. The key to the fast solution. |
| Can values be 0? | **Yes.** | A 0 is a trap: you can't move from it. |
| Only one element? | You're already at the end → `true`. | |
| Return the path or count? | **No**, just true/false. | The count is Question 10. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** from where you stand, **try every allowed jump**, and from each landing spot try every jump again (recursion). If any path reaches the end, answer true.

🗣️ *"The most direct approach tries every possible path recursively: from index i, try every jump length from 1 to nums[i]. If any path reaches the last index, return true."*

**Why is it slow?** The number of paths explodes. Each stone can branch into many jumps, so the total can be exponential. And the same stone gets re-explored from many different starting points, each time redoing the same work.

🗣️ *"In the worst case that's exponential, because the same index is reached by many different paths and I re-solve it every time."*

In [1]:
class SolutionBrute:
    def canJump(self, nums: list[int]) -> bool:
        last = len(nums) - 1

        def can_reach_end_from(i: int) -> bool:
            if i >= last:
                return True
            for step in range(1, nums[i] + 1):      # try every allowed jump
                if can_reach_end_from(i + step):
                    return True
            return False

        return can_reach_end_from(0)

#### Step 1½ · Remember answers (dynamic programming)

Each stone's answer ("can I finish from here?") never changes, so compute it **once**. Go **right to left**: a stone is "good" if any stone it can jump to is good.

🗣️ *"Each index's answer depends only on indices to its right, so I can fill a 'good' array from the end. That's O(n) stones, but each checks up to nums[i] neighbours, so it's O(n²) in the worst case, about 10⁸ for n = 10⁴. Better, but I think there's a linear solution."*

In [2]:
class SolutionDP:
    def canJump(self, nums: list[int]) -> bool:
        n = len(nums)
        good = [False] * n
        good[-1] = True                              # the last stone is the goal
        for i in range(n - 2, -1, -1):
            farthest = min(i + nums[i], n - 1)
            good[i] = any(good[j] for j in range(i + 1, farthest + 1))
        return good[0]

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** because you can always jump **shorter**, if you can reach stone 7, you can also reach stones 1 to 6. The reachable stones are **always one solid block `[0 … farthest]`**. So I don't need to track *which* stones are reachable, just **one number: how far the block extends**.

Walk left to right:
- If the current index `i` is **beyond** `farthest`, there's a gap: you can never get here → **false**.
- Otherwise you can stand here, so extend the block: `farthest = max(farthest, i + nums[i])`.
- If `farthest` reaches the last index → **true** (you can stop early).

🗣️ *"Because each value is a maximum, the set of reachable indices is always a contiguous range starting at 0. So I only track its right edge, 'farthest'. I scan left to right: if my current index is beyond farthest, I can't get there, so return false. Otherwise I update farthest with i + nums[i]. If farthest reaches the last index, return true. One pass, O(n) time, O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Try every path (recursion) | ❌ | Exponential. Re-explores the same stones again and again. |
| BFS with a queue of stones | ⚠️ | Visits each stone once, but pushing all neighbours can still be O(n²) work, plus O(n) memory. |
| DP "good" array | ⚠️ | No repeated work, but the inner "any good neighbour?" loop keeps it O(n²), with O(n) memory. |
| **One `farthest` number** | ✅ **best** | Uses the "no gaps" fact. One pass, constant memory, no data structure at all. |
| Backwards greedy: move a `goal` left whenever `i + nums[i] ≥ goal` | ✅ (alternative) | Equally good, O(n)/O(1). Answer is `goal == 0`. Nice to mention. |

**Why is "greedy" safe here?** We never actually **choose** a jump. `farthest` summarises **every** possible path at once, so there's no wrong decision to regret.

In [3]:
class Solution:
    def canJump(self, nums: list[int]) -> bool:
        farthest = 0                     # right edge of the reachable block [0..farthest]
        last = len(nums) - 1
        for i, jump in enumerate(nums):
            if i > farthest:             # gap: index i can't be reached
                return False
            farthest = max(farthest, i + jump)
            if farthest >= last:         # the goal is inside the block
                return True
        return True

### Step 3 · Toy example walkthrough (tell it like a story)

**Example A:** `nums = [2, 3, 1, 1, 4]`, last index = 4

| i | nums[i] | i > farthest? | i + nums[i] | farthest after | note |
|---|---|---|---|---|---|
| 0 | 2 | 0 > 0 ❌ | 2 | 2 | reachable block is 0..2 |
| 1 | 3 | 1 > 2 ❌ | 4 | **4** | 4 ≥ last → **True** ✅ |

**Narrated:** "From stone 0 I can reach up to stone 2. Stone 1 is inside that, and from stone 1 I can reach stone 4, the end. Done, without even looking at the rest."

**Example B:** `nums = [3, 2, 1, 0, 4]`

| i | nums[i] | i > farthest? | i + nums[i] | farthest after |
|---|---|---|---|---|
| 0 | 3 | ❌ | 3 | 3 |
| 1 | 2 | ❌ | 3 | 3 |
| 2 | 1 | ❌ | 3 | 3 |
| 3 | 0 | ❌ | 3 | 3 |
| 4 | 4 | **4 > 3 → gap** | | **False** ✅ |

**Narrated:** "Every stone I can reach only gets me as far as stone 3, and stone 3 is a zero. When I get to stone 4, it's outside my reachable block, so it's impossible."

In [4]:
cases = [
    ([2, 3, 1, 1, 4], True),
    ([3, 2, 1, 0, 4], False),
    ([0], True),               # already at the end
    ([0, 1], False),           # stuck at the start
    ([1, 0, 1, 0], False),
    ([2, 0, 0], True),         # jump over the zeros
    ([1, 1, 1, 1], True),
]
for nums, expected in cases:
    for Impl in (SolutionBrute, SolutionDP, Solution):
        assert Impl().canJump(nums) == expected, Impl.__name__
    print(nums, "->", expected)

import random
for _ in range(500):
    nums = [random.randint(0, 3) for _ in range(random.randint(1, 10))]
    assert Solution().canJump(nums) == SolutionBrute().canJump(nums), nums
print("All tests passed ✅")

[2, 3, 1, 1, 4] -> True
[3, 2, 1, 0, 4] -> False
[0] -> True
[0, 1] -> False
[1, 0, 1, 0] -> False
[2, 0, 0] -> True
[1, 1, 1, 1] -> True
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Try every path | O(2ⁿ) worst | O(n) call stack | Explodes as the array grows. |
| DP "good" array | O(n²) | O(n) | Each stone checks all its neighbours. |
| **Farthest reach** | **O(n)** | **O(1)** | Each stone is looked at once; one number is remembered. |

**Why O(n) time?** One loop, with a comparison and a `max` per stone.

**Why O(1) memory?** Only `farthest` and `last`, regardless of the input size.

**Edge cases to mention:** a single element (true), a 0 at the start with more stones after it (false), zeros you can jump *over* (`[2,0,0]` → true).

**Likely follow-ups:**
- *Fewest jumps to reach the end?* → Question 10. The same "farthest" idea, counted in rounds.
- *Show the actual path?* → Remember, for each stone, which earlier stone first reached it.

---

#### 🗣️ Full interview script (about 60 seconds)

*"Each value is the maximum jump from that index, and I need to know whether I can reach the last index.*

*Trying every path recursively is exponential. Caching which indices can reach the end gives O(n²) DP.*

*The key insight is that since jumps are maximums, the reachable indices always form one contiguous block starting at 0. So I only track its right edge, 'farthest'. Scanning left to right, if the current index is past farthest, there's a gap and I return false. Otherwise I extend farthest to i + nums[i], and as soon as farthest covers the last index I return true.*

*That's one pass, O(n) time and O(1) space. It's safe to be greedy because farthest summarises all possible paths at once; I never commit to a specific jump."*